# 22. PatchCore (model, trained on good tiles only)

A frozen ImageNet-pretrained network describes small patches of good fabric; a memory bank keeps a representative subset of them (coreset). Anomaly = distance of each patch of a tile to its nearest good patches. No training loop, no defect labels.

Model: `models/patchcore.py` (`METHOD_NAME = "patchcore"`). Evaluation: `evaluate.py`, same rules and same test tiles as the classical experiments (README section 7). The model is fitted on part of the `good_cal` strips and the threshold comes from the other `good_cal` strips (`FIT_FRACTION`): a model scores the tiles it was fitted on as "too normal", so they cannot also set the threshold. Per-tile scores and the split are saved to `results/patchcore/`.

Needs `torch` and `torchvision`. Uses a GPU if there is one; on a CPU it is slower (the first cell prints which).

In [ ]:
import sys, importlib

sys.path.insert(0, "../src/methods")
import filters as F    # tile score, threshold and display helpers
import evaluate as E   # shared evaluation (README section 7)
import models as M     # PatchCore and PaDiM (needs torch + torchvision)
importlib.reload(F); importlib.reload(E); importlib.reload(M)  # pick up edits without restarting the kernel
print(M.device_info())

# ---- evaluation settings (same for every experiment) ----------------------
FABRIC = "02"           # fabric code = third part of the file name
METHOD_NAME = "patchcore"  # results go to results/<METHOD_NAME>/
THR_K = 4.0             # threshold = median + THR_K x 1.4826 x MAD of good_cal scores
MIN_AREA = 4            # damage mask: drop spots smaller than this (px); keep <= 4
SCORE_K = 25            # tile score = mean of the SCORE_K highest values of the smoothed map
SCORE_SIGMA = 2.0       # smoothing before scoring (px)
NORMALIZE = True        # score = strongest spot / the tile's own typical level (removes strip-to-strip bias)
FIT_FRACTION = 0.7      # models only: this share of the good_cal strips trains the model, the rest sets the threshold
SETTINGS = dict(thr_k=THR_K, min_area=MIN_AREA, score_k=SCORE_K, score_sigma=SCORE_SIGMA,
                normalize=NORMALIZE, fit_fraction=FIT_FRACTION)

# ---- model settings (hyperparameters, explained in models/patchcore.py) -------------
PARAMS = dict(
    backbone="wide_resnet50_2",    # paper's network; "resnet18" is about 5x faster and less accurate
    layers=("layer2", "layer3"),   # feature maps used (32 x 32 patch grid on a 256 px tile)
    coreset=0.01,                  # share of good patches kept in the memory bank
    n_neighbors=1,                 # anomaly = distance to this many nearest good patches
    pool=3,                        # local averaging of features
    smooth=4.0,                    # Gaussian sigma (px) of the final map
    tile_norm=False,               # features relative to the tile's own average patch
)
method = None


def make_fit(**params):
    def fit(good_tiles):
        """Memory bank from good tiles only."""
        fitted = M.fit_patchcore(good_tiles, **params)
        return lambda tile: M.patchcore_map(tile, fitted)
    return fit


fit = make_fit(**PARAMS)

## Results for one fabric

In [ ]:
res = E.evaluate(method, FABRIC, fit=fit, **SETTINGS)
assert res is not None, f"not enough tiles for fabric {FABRIC}"
S = E.summarize(res)
E.print_summary(res, S)
print("saved to", E.save_results(res, METHOD_NAME))

### Score distribution and ROC curve
Left: tile scores with the threshold. Right: recall vs false alarm rate for every possible threshold; the dot is the threshold chosen from good_cal. The curve uses test tiles and is for analysis only.

In [ ]:
E.plot_scores_roc(res, S)

### Recall per defect type

In [ ]:
E.print_by_type(res)

### Failure cases
Missed defects (lowest-scoring defect tiles) and false alarms (highest-scoring good_test / hard-negative tiles). Red = detected, green = ground-truth mask.

In [ ]:
E.show_failures(res, n=3)

### Example tiles
Largest defect, smallest defect (by mask area) and the last good tile of `FABRIC`.

In [ ]:
E.show_examples(res)

### One example per defect type
The dataset has 12 defect types (second part of the file name), spread over different fabrics. One tile per type: the clearest one (largest mask) from the fabric with the most tiles of that type. Each is judged by a model fitted and thresholded on its own fabric's good_cal strips (other fabrics are fitted here, so this cell takes longer). Red = detected, green = ground-truth mask.

In [ ]:
E.show_per_type(method, fit, res, **SETTINGS)

## All fabrics
Same settings on every fabric, one model and one threshold per fabric from its own good_cal strips. Columns: FAR on good_test and on hard negatives (hn), recall overall / small / large defects, strip-level FAR and recall, AUROC, IoU, ms per tile.

In [ ]:
summaries = E.all_fabrics(method, fit, METHOD_NAME, **SETTINGS)

## Hyperparameter tuning (optional)
Compares changes to `PARAMS` on `FABRIC` only (the development fabric). Copy the best row's values into `PARAMS` in the first code cell and rerun.

Picking a setting from this table uses the defect tiles of `FABRIC`, so the numbers of that fabric become optimistic. Judge the chosen setting on the **other** fabrics (the "All fabrics" table). Do not tune on every fabric separately.

In [ ]:
RUN_TUNING = False   # True = run the comparison below (each row is a full run on FABRIC)
GRID = [             # each entry = a change to PARAMS
    {},
    {"backbone": "resnet18"},
    {"layers": ("layer1", "layer2"), "smooth": 2.0, "max_patches": 60000, "coreset": 0.02},
    {"coreset": 0.1},
    {"n_neighbors": 3},
    {"smooth": 2.0},
    {"tile_norm": True},
]
if RUN_TUNING:
    tuning = E.tune(make_fit, PARAMS, GRID, FABRIC, **SETTINGS)